## User-Defined Tool with the Claude API

### Installing Utilities and Libraries

In [1]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [3]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Define the Tool Schema

In [4]:
tools = [
    {
        "name": "GetCurrentWeather",
        "description": "Retrieve the current weather information for a specified location using the wttr.in weather service.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City or location to retrieve the weather for, for example London, New York, or Mumbai."
                }
            },
            "required": ["location"]
        }
    }
]

### Implement the Weather API Call

In [5]:
import requests

def get_current_weather(location: str):

    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return response.json()

### Implement the Agent Loop

In [6]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in London today?"
    }
]

while True:
    response = client.messages.create(
        model = claude_model_name,
        max_tokens = 4096,
        tools = tools,
        tool_choice = {"type": "auto"},
        messages = messages
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content
        }
    )

    tool_used = False

    for block in response.content:
        if block.type == "text":
            print(block.text + "\n\n")

        elif block.type == "tool_use":

            tool_used = True

            if block.name == "GetCurrentWeather":

                print("Invoking GetCurrentWeather Tool \n\n")
                weather = get_current_weather(
                    block.input["location"]
                )

                messages.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": str(weather)
                            }
                        ]
                    }
                )

    if not tool_used:
        break

Invoking GetCurrentWeather Tool 


Here's the current weather in London:

**☁️ Overcast, 18°C (64°F)** — feels like 15°C (59°F)

- **Humidity:** 48%
- **Wind:** 12 km/h (8 mph) from the WNW
- **Visibility:** 10 km (6 miles)
- **Pressure:** 1018 mb
- **UV Index:** 0
- **Precipitation:** None currently

**Today's range:** High of 18°C (65°F), low of 13°C (55°F), with about 6 hours of sunshine expected. There's a chance of some light rain showers later this evening, but nothing heavy expected.

Overall, a cloudy but mild day in London — good for a light jacket, no umbrella strictly necessary but worth carrying one just in case! ☂️


